# LegalRAG — Restartable Kaggle Pipeline

Hybrid-retrieval RAG over Indian High Court judgments.

Pipeline stages:
1. Configuration & environment validation
2. Artifact/path helpers
3. Corpus loading (cleaned judgments)
4. Chunk loading (baseline chunks)
5. Gold evaluation set loading
6. BM25 index build/load + evaluation
7. Dense embedding model load
8. Checkpointed embedding generation (sharded)
9. FAISS index build/load
10. Dense retrieval + evaluation
11. Experiment results summary

Every stage is idempotent: checks for existing artifacts, validates, skips if complete, otherwise computes and persists.

In [ ]:
# ============================================================
# 1. CONFIGURATION — Centralized paths & hyperparameters
# ============================================================
import os
import json
import sys
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional

@dataclass(frozen=True)
class Config:
    # Root directories (Kaggle-standard)
    root: Path = Path("/kaggle/working")
    data_raw: Path = Path("/kaggle/working/data/raw")
    data_cleaned: Path = Path("/kaggle/working/data/cleaned")
    data_chunks: Path = Path("/kaggle/working/data/chunks")
    evaluation_dir: Path = Path("/kaggle/working/evaluation")
    bm25_dir: Path = Path("/kaggle/working/bm25")
    embeddings_dir: Path = Path("/kaggle/working/embeddings")
    faiss_dir: Path = Path("/kaggle/working/faiss")
    results_dir: Path = Path("/kaggle/working/results")

    # Corpus artifacts
    cleaned_corpus_path: Path = Path("/kaggle/working/data/cleaned/cleaned_corpus.parquet")
    chunks_path: Path = Path("/kaggle/working/data/chunks/chunks.parquet")
    gold_eval_path: Path = Path("/kaggle/working/evaluation/gold_eval.json")

    # BM25 artifacts
    bm25_index_path: Path = Path("/kaggle/working/bm25/bm25_index.pkl")
    bm25_tokenized_path: Path = Path("/kaggle/working/bm25/tokenized_corpus.pkl")
    bm25_results_path: Path = Path("/kaggle/working/results/bm25_top10.json")
    bm25_metrics_path: Path = Path("/kaggle/working/results/bm25_metrics.json")

    # Embedding artifacts
    embedding_model_name: str = "BAAI/bge-small-en-v1.5"
    embedding_dim: int = 384
    shard_size: int = 5000  # chunks per embedding shard
    embedding_batch_size: int = 64
    normalize_embeddings: bool = True

    # FAISS artifacts
    faiss_index_path: Path = Path("/kaggle/working/faiss/faiss_index.ipx")
    faiss_meta_path: Path = Path("/kaggle/working/faiss/faiss_meta.json")

    # Dense retrieval artifacts
    dense_results_path: Path = Path("/kaggle/working/results/dense_top10.json")
    dense_metrics_path: Path = Path("/kaggle/working/results/dense_metrics.json")

    # Chunking parameters (MUST match baseline)
    chunk_size: int = 1200
    chunk_overlap: int = 200
    min_chunk_length: int = 100

    # Evaluation
    eval_top_k: int = 10
    recall_ks: List[int] = None

    def __post_init__(self):
        if self.recall_ks is None:
            object.__setattr__(self, 'recall_ks', [1, 3, 5, 10])
        # Ensure all directories exist
        for p in [
            self.data_raw, self.data_cleaned, self.data_chunks,
            self.evaluation_dir, self.bm25_dir, self.embeddings_dir,
            self.faiss_dir, self.results_dir
        ]:
            p.mkdir(parents=True, exist_ok=True)

CFG = Config()

print("Configuration loaded")
print(f"  Root: {CFG.root}")
print(f"  Cleaned corpus: {CFG.cleaned_corpus_path}")
print(f"  Chunks: {CFG.chunks_path}")
print(f"  Gold eval: {CFG.gold_eval_path}")
print(f"  Embedding model: {CFG.embedding_model_name}")
print(f"  Shard size: {CFG.shard_size}")

# 2. ENVIRONMENT VALIDATION

Verify GPU availability, Python packages, and disk space.

In [ ]:
# ============================================================
# 2. ENVIRONMENT VALIDATION
# ============================================================
import torch
import subprocess
import shutil

print("=== Environment Validation ===")

# Python version
print(f"Python: {sys.version.split()[0]}")

# GPU
if torch.cuda.is_available():
    print(f"GPUs available: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  cuda:{i} — {props.name} ({props.total_memory / 1e9:.1f} GB)")
    CFG_GPU_COUNT = torch.cuda.device_count()
else:
    print("GPU: NOT AVAILABLE (CPU only)")
    CFG_GPU_COUNT = 0

# Disk space
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Disk: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

# Key packages
import pandas as pd
import numpy as np
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"torch: {torch.__version__}")

# Verify required packages
required = ["sentence_transformers", "faiss", "rank_bm25", "tqdm"]
for pkg in required:
    try:
        __import__(pkg)
        print(f"  {pkg}: OK")
    except ImportError:
        print(f"  {pkg}: MISSING — will install")

# 3. ARTIFACT HELPERS

Utility functions for checking, validating, and loading/saving artifacts.

In [ ]:
# ============================================================
# 3. ARTIFACT HELPERS
# ============================================================
import pickle
import hashlib
from pathlib import Path
from typing import Any, Callable, TypeVar

T = TypeVar('T')

def file_exists_and_nonempty(path: Path) -> bool:
    """Check if file exists and has non-zero size."""
    return path.exists() and path.stat().st_size > 0

def compute_sha256(path: Path) -> str:
    """Compute SHA256 of a file for integrity checks."""
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

def save_json(obj: Any, path: Path) -> None:
    """Atomic JSON save."""
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2)
    tmp.replace(path)

def load_json(path: Path) -> Any:
    with open(path, 'r') as f:
        return json.load(f)

def save_pickle(obj: Any, path: Path) -> None:
    """Atomic pickle save."""
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'wb') as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    tmp.replace(path)

def load_pickle(path: Path) -> Any:
    with open(path, 'rb') as f:
        return pickle.load(f)

def save_npy(arr: 'np.ndarray', path: Path) -> None:
    """Atomic numpy save."""
    tmp = path.with_suffix(path.suffix + '.tmp')
    np.save(tmp, arr)
    tmp.replace(path)

def load_npy(path: Path) -> 'np.ndarray':
    return np.load(path, mmap_mode='r')

def artifact_valid(path: Path, validator: Callable[[Any], bool] = None) -> bool:
    """Check if artifact exists and optionally passes validator."""
    if not file_exists_and_nonempty(path):
        return False
    if validator is None:
        return True
    try:
        return validator(path)
    except Exception:
        return False

def get_shard_paths(base_dir: Path, prefix: str, total_shards: int) -> List[Path]:
    """Generate list of shard paths: emb_000.npy, emb_001.npy, ..."""
    return [base_dir / f"{prefix}_{i:03d}.npy" for i in range(total_shards)]

def count_shards(base_dir: Path, prefix: str) -> int:
    """Count existing valid shard files."""
    count = 0
    for p in sorted(base_dir.glob(f"{prefix}_*.npy")):
        if file_exists_and_nonempty(p):
            count += 1
    return count

print("Artifact helpers ready")

# 4. CORPUS LOADING

Load the cleaned corpus (49,629 judgments). This artifact should already exist from the data collection phase. If missing, the notebook will error clearly rather than silently regenerating.

In [ ]:
# ============================================================
# 4. CORPUS LOADING
# ============================================================
import pandas as pd
import gc

print("=== Loading Cleaned Corpus ===")

if not file_exists_and_nonempty(CFG.cleaned_corpus_path):
    raise FileNotFoundError(
        f"Cleaned corpus not found at {CFG.cleaned_corpus_path}. "
        f"Run the data collection/cleaning pipeline first."
    )

corpus_df = pd.read_parquet(CFG.cleaned_corpus_path)
print(f"Loaded {len(corpus_df)} judgments")
print(f"Columns: {list(corpus_df.columns)}")
print(f"Memory: {corpus_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# Basic validation
assert 'cnr' in corpus_df.columns, "Missing 'cnr' column"
assert 'clean_text' in corpus_df.columns, "Missing 'clean_text' column"
assert corpus_df['cnr'].is_unique, "CNR values are not unique"
print("Corpus validation passed")

# Keep only needed columns to save memory
corpus_df = corpus_df[['cnr', 'clean_text']].copy()
gc.collect()

# 5. CHUNK LOADING

Load baseline chunks (303,734 chunks). Chunking parameters are fixed: size=1200, overlap=200, min=100. Do not rechunk — load the persisted artifact.

In [ ]:
# ============================================================
# 5. CHUNK LOADING
# ============================================================
import pandas as pd
import gc

print("=== Loading Baseline Chunks ===")

if not file_exists_and_nonempty(CFG.chunks_path):
    raise FileNotFoundError(
        f"Chunks not found at {CFG.chunks_path}. "
        f"Run the chunking pipeline first."
    )

chunks_df = pd.read_parquet(CFG.chunks_path)
print(f"Loaded {len(chunks_df)} chunks")
print(f"Columns: {list(chunks_df.columns)}")
print(f"Memory: {chunks_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# Validation
assert 'chunk_id' in chunks_df.columns, "Missing 'chunk_id' column"
assert 'cnr' in chunks_df.columns, "Missing 'cnr' column"
assert 'text' in chunks_df.columns, "Missing 'text' column"
assert chunks_df['chunk_id'].is_unique, "chunk_id values are not unique"

# Verify chunking parameters match baseline
expected_chunks = 303734
if len(chunks_df) != expected_chunks:
    print(f"WARNING: Expected {expected_chunks} chunks, got {len(chunks_df)}")

print("Chunk validation passed")

# Keep only needed columns
chunks_df = chunks_df[['chunk_id', 'cnr', 'text']].copy()
gc.collect()

# 6. GOLD EVALUATION SET LOADING

Load the 100-question grounded evaluation set. This artifact is FIXED — do not regenerate. If missing, error clearly.

In [ ]:
# ============================================================
# 6. GOLD EVALUATION SET LOADING
# ============================================================
import json
from collections import Counter

print("=== Loading Gold Evaluation Set ===")

if not file_exists_and_nonempty(CFG.gold_eval_path):
    raise FileNotFoundError(
        f"Gold evaluation set not found at {CFG.gold_eval_path}. "
        f"This artifact must be created by the evaluation generation pipeline. "
        f"Do not regenerate — the benchmark is fixed."
    )

with open(CFG.gold_eval_path, 'r') as f:
    gold_eval = json.load(f)

print(f"Loaded {len(gold_eval)} evaluation questions")

# Validation
required_keys = {'cnr', 'question', 'reference_answer', 'question_type', 'supporting_text', 'gold_chunk_ids'}
for i, item in enumerate(gold_eval):
    missing = required_keys - set(item.keys())
    if missing:
        raise ValueError(f"Item {i} missing keys: {missing}")
    if not item['gold_chunk_ids']:
        raise ValueError(f"Item {i} (cnr={item['cnr']}) has empty gold_chunk_ids")
    for chunk_id in item['gold_chunk_ids']:
        if chunk_id not in chunks_df['chunk_id'].values:
            print(f"WARNING: gold chunk {chunk_id} not found in chunks_df")

type_counts = Counter(item['question_type'] for item in gold_eval)
print(f"Question type distribution: {dict(type_counts)}")
print(f"Questions with gold evidence: {sum(bool(x['gold_chunk_ids']) for x in gold_eval)}/{len(gold_eval)}")

# Sanity check: should match baseline distribution
expected_dist = {'reasoning': 34, 'outcome': 27, 'legal_provision': 21, 'fact': 18}
for k, v in expected_dist.items():
    actual = type_counts.get(k, 0)
    if actual != v:
        print(f"  NOTE: {k}: expected {v}, got {actual}")

print("Gold evaluation set validation passed")

# 7. BM25 INDEX — BUILD OR LOAD

Build BM25 index if not present, otherwise load. The tokenized corpus is large (~300k docs) so we persist it. BM25 runs on CPU.

In [ ]:
# ============================================================
# 7. BM25 INDEX — BUILD OR LOAD
# ============================================================
import pickle
import gc
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

print("=== BM25 Index ===")

def build_bm25_index() -> BM25Okapi:
    """Build BM25 index from chunks_df text. Returns BM25Okapi instance."""
    print("Tokenizing corpus...")
    corpus_texts = chunks_df['text'].tolist()
    
    # Tokenize in batches to control memory
    tokenized = []
    batch_size = 10000
    for i in tqdm(range(0, len(corpus_texts), batch_size), desc="Tokenizing"):
        batch = corpus_texts[i:i+batch_size]
        tokenized.extend([t.lower().split() for t in batch])
    
    print(f"Tokenized {len(tokenized)} documents")
    
    # Save tokenized corpus for reuse
    save_pickle(tokenized, CFG.bm25_tokenized_path)
    
    print("Building BM25 index...")
    bm25 = BM25Okapi(tokenized)
    
    # Save index
    save_pickle(bm25, CFG.bm25_index_path)
    
    # Release tokenized corpus from memory (BM25 keeps its own copy)
    del tokenized
    gc.collect()
    
    return bm25

def load_bm25_index() -> BM25Okapi:
    """Load persisted BM25 index."""
    print("Loading BM25 index from disk...")
    return load_pickle(CFG.bm25_index_path)

# Check if valid artifacts exist
index_valid = artifact_valid(CFG.bm25_index_path)
tokenized_valid = artifact_valid(CFG.bm25_tokenized_path)

if index_valid and tokenized_valid:
    bm25 = load_bm25_index()
    print("BM25 index loaded from cache")
else:
    print("Building BM25 index (first run or invalid cache)...")
    bm25 = build_bm25_index()
    print("BM25 index built and cached")

print(f"BM25 corpus size: {len(bm25.corpus_size)}")
print(f"BM25 avgdl: {bm25.avgdl:.1f}")

# 8. BM25 EVALUATION

Run BM25 retrieval on all 100 questions, retrieve top-10 once per question, derive Recall@1,3,5,10. Save results for reuse.

In [ ]:
# ============================================================
# 8. BM25 EVALUATION
# ============================================================
import numpy as np
from tqdm.auto import tqdm

print("=== BM25 Evaluation ===")

def bm25_search(query: str, k: int = 10) -> pd.DataFrame:
    """Retrieve top-k chunks for a query using BM25."""
    query_tokens = query.lower().split()
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argpartition(scores, -k)[-k:]
    top_indices = top_indices[np.argsort(scores[top_indices])][::-1]
    return chunks_df.iloc[top_indices].copy()

# Check if results already exist
if file_exists_and_nonempty(CFG.bm25_results_path):
    print("Loading cached BM25 top-10 results...")
    bm25_top10 = load_json(CFG.bm25_results_path)
    # Convert to DataFrames
    bm25_top10_dfs = {cnr: pd.DataFrame(rows) for cnr, rows in bm25_top10.items()}
else:
    print("Running BM25 retrieval for all evaluation questions...")
    bm25_top10_dfs = {}
    
    for item in tqdm(gold_eval, desc="BM25 retrieval"):
        cnr = item['cnr']
        retrieved = bm25_search(item['question'], k=CFG.eval_top_k)
        bm25_top10_dfs[cnr] = retrieved
    
    # Persist as JSON-serializable records
    bm25_top10_serializable = {
        cnr: df[['chunk_id', 'cnr', 'text']].to_dict('records')
        for cnr, df in bm25_top10_dfs.items()
    }
    save_json(bm25_top10_serializable, CFG.bm25_results_path)
    print(f"Saved BM25 top-10 results to {CFG.bm25_results_path}")

# Compute Recall@K from saved top-10 results
print("\nComputing Recall@K...")
bm25_metrics = {}
for k in CFG.recall_ks:
    scores = []
    for item in gold_eval:
        retrieved = bm25_top10_dfs[item['cnr']].head(k)
        retrieved_ids = set(retrieved['chunk_id'])
        gold_ids = set(item['gold_chunk_ids'])
        scores.append(int(bool(retrieved_ids & gold_ids)))
    recall = np.mean(scores)
    bm25_metrics[f'recall@{k}'] = float(recall)
    print(f"  Recall@{k}: {recall:.4f}")

# Save metrics
bm25_metrics_output = {
    'retriever': 'BM25',
    **bm25_metrics
}
save_json(bm25_metrics_output, CFG.bm25_metrics_path)
print(f"\nBM25 metrics saved to {CFG.bm25_metrics_path}")

# Verify against baseline
baseline = {'recall@1': 0.20, 'recall@3': 0.24, 'recall@5': 0.28, 'recall@10': 0.33}
print("\nBaseline comparison:")
for k in CFG.recall_ks:
    key = f'recall@{k}'
    print(f"  {key}: computed={bm25_metrics[key]:.4f}, baseline={baseline[key]:.4f}, diff={bm25_metrics[key]-baseline[key]:+.4f}")

# 9. DENSE EMBEDDING MODEL LOAD

Load the BGE-small-en-v1.5 model on GPU. This is the fixed baseline model — do not change.

In [ ]:
# ============================================================
# 9. DENSE EMBEDDING MODEL LOAD
# ============================================================
import torch
from sentence_transformers import SentenceTransformer

print("=== Loading Embedding Model ===")

def load_embedding_model() -> SentenceTransformer:
    """Load SentenceTransformer on available GPUs."""
    if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
        # Use multi-process pool for multi-GPU encoding later
        device = "cuda"
    elif torch.cuda.is_available():
        device = "cuda:0"
    else:
        device = "cpu"
    
    model = SentenceTransformer(CFG.embedding_model_name, device=device)
    print(f"Model loaded on {device}")
    print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")
    return model

# Load model (lightweight, fast)
embedding_model = load_embedding_model()

# Verify model identity
assert embedding_model.get_sentence_embedding_dimension() == CFG.embedding_dim, \
    f"Model dimension mismatch: expected {CFG.embedding_dim}"

# 10. CHECKPOINTED EMBEDDING GENERATION

Generate embeddings in shards of 5,000 chunks. Each shard saved as `emb_XXX.npy`.
Resumes automatically: existing valid shards are skipped.
Uses multi-GPU via `encode_multi_process` when available.
Memory-safe: encodes one shard at a time, releases memory immediately.

In [ ]:
# ============================================================
# 10. CHECKPOINTED EMBEDDING GENERATION
# ============================================================
import numpy as np
import gc
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

print("=== Checkpointed Embedding Generation ===")

total_chunks = len(chunks_df)
num_shards = (total_chunks + CFG.shard_size - 1) // CFG.shard_size
print(f"Total chunks: {total_chunks}")
print(f"Shard size: {CFG.shard_size}")
print(f"Total shards: {num_shards}")

def shard_path(shard_idx: int) -> Path:
    return CFG.embeddings_dir / f"emb_{shard_idx:03d}.npy"

def shard_meta_path(shard_idx: int) -> Path:
    return CFG.embeddings_dir / f"emb_{shard_idx:03d}.meta.json"

def validate_shard(shard_idx: int) -> bool:
    """Check if shard file exists, has correct shape, and matches metadata."""
    p = shard_path(shard_idx)
    m = shard_meta_path(shard_idx)
    if not (file_exists_and_nonempty(p) and file_exists_and_nonempty(m)):
        return False
    try:
        meta = load_json(m)
        arr = load_npy(p)
        expected_rows = meta.get('num_chunks', 0)
        expected_dim = meta.get('embedding_dim', 0)
        if arr.shape != (expected_rows, expected_dim):
            return False
        # Verify model name matches
        if meta.get('model_name') != CFG.embedding_model_name:
            return False
        return True
    except Exception:
        return False

def encode_shard(shard_idx: int, model: SentenceTransformer) -> None:
    """Encode a single shard and save atomically with metadata."""
    start = shard_idx * CFG.shard_size
    end = min(start + CFG.shard_size, total_chunks)
    chunk_texts = chunks_df.iloc[start:end]['text'].tolist()
    
    # Encode with multi-GPU if available
    if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
        devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())]
        pool = model.start_multi_process_pool(target_devices=devices)
        try:
            embeddings = model.encode_multi_process(
                chunk_texts,
                pool=pool,
                batch_size=CFG.embedding_batch_size,
                normalize_embeddings=CFG.normalize_embeddings,
                show_progress_bar=False
            )
        finally:
            model.stop_multi_process_pool(pool)
    else:
        embeddings = model.encode(
            chunk_texts,
            batch_size=CFG.embedding_batch_size,
            normalize_embeddings=CFG.normalize_embeddings,
            show_progress_bar=False,
            device=model.device
        )
    
    # Validate shape
    assert embeddings.shape == (end - start, CFG.embedding_dim), \
        f"Shape mismatch: got {embeddings.shape}, expected {(end-start, CFG.embedding_dim)}"
    
    # Save shard
    save_npy(embeddings.astype(np.float32), shard_path(shard_idx))
    
    # Save metadata
    meta = {
        'shard_idx': shard_idx,
        'start_idx': start,
        'end_idx': end,
        'num_chunks': end - start,
        'embedding_dim': CFG.embedding_dim,
        'model_name': CFG.embedding_model_name,
        'normalized': CFG.normalize_embeddings,
        'sha256': compute_sha256(shard_path(shard_idx))
    }
    save_json(meta, shard_meta_path(shard_idx))
    
    # Explicit cleanup
    del embeddings, chunk_texts
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Find first incomplete shard
existing_valid = 0
for i in range(num_shards):
    if validate_shard(i):
        existing_valid += 1
    else:
        break

print(f"Existing valid shards: {existing_valid}/{num_shards}")

if existing_valid == num_shards:
    print("All embedding shards already complete — skipping generation")
else:
    print(f"Generating shards {existing_valid} to {num_shards-1}...")
    for shard_idx in tqdm(range(existing_valid, num_shards), desc="Encoding shards"):
        encode_shard(shard_idx, embedding_model)
    print("All embedding shards generated and saved")

# 11. FAISS INDEX — BUILD OR LOAD

Build FAISS index from all embedding shards. Uses Inner Product (IndexFlatIP) since embeddings are normalized.
Persists index + metadata (model name, chunk count, dimension) for validation on reload.

In [ ]:
# ============================================================
# 11. FAISS INDEX — BUILD OR LOAD
# ============================================================
import faiss
import numpy as np
import gc
from tqdm.auto import tqdm

print("=== FAISS Index ===")

def validate_faiss_index() -> bool:
    """Check if FAISS index exists and metadata matches current config."""
    if not (file_exists_and_nonempty(CFG.faiss_index_path) and file_exists_and_nonempty(CFG.faiss_meta_path)):
        return False
    try:
        meta = load_json(CFG.faiss_meta_path)
        if meta.get('model_name') != CFG.embedding_model_name:
            return False
        if meta.get('embedding_dim') != CFG.embedding_dim:
            return False
        if meta.get('num_chunks') != len(chunks_df):
            return False
        # Try loading index
        index = faiss.read_index(str(CFG.faiss_index_path))
        if index.ntotal != len(chunks_df):
            return False
        return True
    except Exception:
        return False

def build_faiss_index() -> faiss.Index:
    """Build FAISS IndexFlatIP from all embedding shards."""
    print("Building FAISS index from embedding shards...")
    
    index = faiss.IndexFlatIP(CFG.embedding_dim)
    
    num_shards = (len(chunks_df) + CFG.shard_size - 1) // CFG.shard_size
    
    for shard_idx in tqdm(range(num_shards), desc="Adding shards to FAISS"):
        p = shard_path(shard_idx)
        if not file_exists_and_nonempty(p):
        raise FileNotFoundError(f"Missing embedding shard: {p}")
        
        # Load with mmap to avoid memory pressure
        shard_embeddings = load_npy(p)
        index.add(shard_embeddings)
        
        # Explicit cleanup
        del shard_embeddings
    
    print(f"FAISS index built: {index.ntotal} vectors")
    
    # Save index
    faiss.write_index(index, str(CFG.faiss_index_path))
    
    # Save metadata
    meta = {
        'model_name': CFG.embedding_model_name,
        'embedding_dim': CFG.embedding_dim,
        'num_chunks': len(chunks_df),
        'index_type': 'IndexFlatIP',
        'metric': 'inner_product',
        'normalized_embeddings': CFG.normalize_embeddings
    }
    save_json(meta, CFG.faiss_meta_path)
    
    return index

if validate_faiss_index():
    print("Loading FAISS index from cache...")
    faiss_index = faiss.read_index(str(CFG.faiss_index_path))
    print(f"FAISS index loaded: {faiss_index.ntotal} vectors")
else:
    print("Building FAISS index (first run or invalid cache)...")
    faiss_index = build_faiss_index()
    print("FAISS index built and cached")

# Optionally move to GPU if available and beneficial
if torch.cuda.is_available() and faiss.get_num_gpus() > 0:
    print("Moving FAISS index to GPU...")
    res = faiss.StandardGpuResources()
    faiss_index = faiss.index_cpu_to_gpu(res, 0, faiss_index)
    print("FAISS index on GPU")

# 12. DENSE RETRIEVAL

Run dense retrieval on all 100 evaluation questions using the FAISS index.
Retrieve top-10 once per question, save results, compute Recall@1,3,5,10.

In [ ]:
# ============================================================
# 12. DENSE RETRIEVAL
# ============================================================
import numpy as np
from tqdm.auto import tqdm

print("=== Dense Retrieval ===")

def dense_search(query: str, k: int = 10) -> pd.DataFrame:
    """Retrieve top-k chunks for a query using dense embeddings + FAISS."""
    # Encode query
    query_emb = embedding_model.encode(
        [query],
        normalize_embeddings=CFG.normalize_embeddings,
        show_progress_bar=False
    ).astype(np.float32)
    
    # Search FAISS
    scores, indices = faiss_index.search(query_emb, k)
    
    # Map back to chunk DataFrame
    retrieved = chunks_df.iloc[indices[0]].copy()
    retrieved['score'] = scores[0]
    return retrieved

# Check if results already exist
if file_exists_and_nonempty(CFG.dense_results_path):
    print("Loading cached dense top-10 results...")
    dense_top10 = load_json(CFG.dense_results_path)
    dense_top10_dfs = {cnr: pd.DataFrame(rows) for cnr, rows in dense_top10.items()}
else:
    print("Running dense retrieval for all evaluation questions...")
    dense_top10_dfs = {}
    
    for item in tqdm(gold_eval, desc="Dense retrieval"):
        cnr = item['cnr']
        retrieved = dense_search(item['question'], k=CFG.eval_top_k)
        dense_top10_dfs[cnr] = retrieved
    
    # Persist
    dense_top10_serializable = {
        cnr: df[['chunk_id', 'cnr', 'text', 'score']].to_dict('records')
        for cnr, df in dense_top10_dfs.items()
    }
    save_json(dense_top10_serializable, CFG.dense_results_path)
    print(f"Saved dense top-10 results to {CFG.dense_results_path}")

# Compute Recall@K
print("\nComputing Dense Recall@K...")
dense_metrics = {}
for k in CFG.recall_ks:
    scores = []
    for item in gold_eval:
        retrieved = dense_top10_dfs[item['cnr']].head(k)
        retrieved_ids = set(retrieved['chunk_id'])
        gold_ids = set(item['gold_chunk_ids'])
        scores.append(int(bool(retrieved_ids & gold_ids)))
    recall = np.mean(scores)
    dense_metrics[f'recall@{k}'] = float(recall)
    print(f"  Recall@{k}: {recall:.4f}")

# Save metrics
dense_metrics_output = {
    'retriever': f'Dense ({CFG.embedding_model_name})',
    **dense_metrics
}
save_json(dense_metrics_output, CFG.dense_metrics_path)
print(f"\nDense metrics saved to {CFG.dense_metrics_path}")

# 13. EXPERIMENT RESULTS SUMMARY

Compare BM25 vs Dense retrieval side-by-side. Print formatted table. Save combined results.

In [ ]:
# ============================================================
# 13. EXPERIMENT RESULTS SUMMARY
# ============================================================
import pandas as pd

print("=== Experiment Results Summary ===\n")

# Load metrics (re-load to be safe across restarts)
bm25_metrics = load_json(CFG.bm25_metrics_path)
dense_metrics = load_json(CFG.dense_metrics_path)

# Build comparison table
rows = []
for k in CFG.recall_ks:
    key = f'recall@{k}'
    rows.append({
        'Metric': key.upper(),
        'BM25': bm25_metrics.get(key, 0),
        f'Dense ({CFG.embedding_model_name})': dense_metrics.get(key, 0),
        'Delta': dense_metrics.get(key, 0) - bm25_metrics.get(key, 0)
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False, float_format='%.4f'))

# Per-question-type breakdown
print("\n=== Recall by Question Type ===")
for qtype in sorted(set(item['question_type'] for item in gold_eval)):
    subset = [item for item in gold_eval if item['question_type'] == qtype]
    print(f"\n{qtype.upper()} ({len(subset)} questions):")
    for k in CFG.recall_ks:
        key = f'recall@{k}'
        bm25_scores = []
        dense_scores = []
        for item in subset:
            bm25_ret = bm25_top10_dfs[item['cnr']].head(k)
            dense_ret = dense_top10_dfs[item['cnr']].head(k)
            bm25_ids = set(bm25_ret['chunk_id'])
            dense_ids = set(dense_ret['chunk_id'])
            gold_ids = set(item['gold_chunk_ids'])
            bm25_scores.append(int(bool(bm25_ids & gold_ids)))
            dense_scores.append(int(bool(dense_ids & gold_ids)))
        bm25_r = np.mean(bm25_scores)
        dense_r = np.mean(dense_scores)
        print(f"  Recall@{k}: BM25={bm25_r:.4f}, Dense={dense_r:.4f}, Δ={dense_r-bm25_r:+.4f}")

# Save combined results
combined_results = {
    'bm25': bm25_metrics,
    'dense': dense_metrics,
    'config': {
        'embedding_model': CFG.embedding_model_name,
        'chunk_size': CFG.chunk_size,
        'chunk_overlap': CFG.chunk_overlap,
        'min_chunk_length': CFG.min_chunk_length,
        'num_chunks': len(chunks_df),
        'num_eval_questions': len(gold_eval),
        'eval_top_k': CFG.eval_top_k
    }
}
combined_path = CFG.results_dir / "retrieval_comparison.json"
save_json(combined_results, combined_path)
print(f"\nCombined results saved to {combined_path}")

# 14. MEMORY DIAGNOSTICS (Optional)

Quick memory snapshot after each major stage for monitoring.

In [ ]:
# ============================================================
# 14. MEMORY DIAGNOSTICS
# ============================================================
import psutil
import os

process = psutil.Process(os.getpid())
mem = process.memory_info()
print(f"Process RSS: {mem.rss / 1e9:.2f} GB")
print(f"Process VMS: {mem.vms / 1e9:.2f} GB")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1e9
        reserved = torch.cuda.memory_reserved(i) / 1e9
        print(f"GPU {i}: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")